# 03 - Limpeza e Preparação dos Dados

Trata, tabela por tabela, apenas os problemas identificados em `02_avaliacao_qualidade.ipynb` - sem aplicar
técnicas que os dados não exigem. Cada decisão é justificada na célula de markdown correspondente. O resultado
é salvo em `data/processed/`, mantendo a mesma granularidade dos arquivos originais (um CSV por tabela): a
etapa de **juntar** as tabelas para análise é transformação e fica para as próximas etapas do projeto.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
geo = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
cat_translation = pd.read_csv(RAW / "product_category_name_translation.csv")

## orders

**Decisão:** converter as 5 colunas de data/hora de texto para `datetime`. Os nulos em `order_approved_at`,
`order_delivered_carrier_date` e `order_delivered_customer_date` **não são imputados**: eles ocorrem porque o
pedido nunca passou por aquela etapa (cancelado, indisponível, ainda em processamento/transporte) - preencher
esse valor inventaria um evento que não aconteceu. Nenhuma linha duplicada a remover (confirmado na avaliação).

In [2]:
colunas_data_orders = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
for col in colunas_data_orders:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## items

**Decisão:** converter `shipping_limit_date` para `datetime`. `price` e `freight_value` já são numéricos e sem
valores inválidos (todos > 0 e >= 0, confirmado na avaliação) - nenhuma correção necessária.

In [3]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
items.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

## payments

**Decisões:**
1. Remover as 3 linhas com `payment_type == "not_defined"` e `payment_value == 0`: não representam uma forma de
   pagamento real nem um valor cobrado, e não há como inferir o tipo correto - mantê-las introduziria uma
   categoria inválida em qualquer contagem por `payment_type`.
2. Corrigir `payment_installments == 0` para `1` nas 2 linhas afetadas: zero parcelas é fisicamente impossível
   quando há um valor de pagamento registrado (mesmo um pagamento único conta como 1 parcela neste esquema).
3. **Não remover** as 6 linhas restantes com `payment_value == 0` do tipo `voucher`: são parcelas de pagamentos
   fracionados em múltiplos vouchers (`payment_sequential` > 1) e são legítimas.

In [4]:
linhas_antes = len(payments)

payments = payments[~((payments["payment_type"] == "not_defined") & (payments["payment_value"] == 0))].copy()
print(f"Removidas {linhas_antes - len(payments)} linhas (not_defined + valor 0)")

payments.loc[payments["payment_installments"] == 0, "payment_installments"] = 1

print("payment_installments == 0 após correção:", (payments["payment_installments"] == 0).sum())
print("payment_type após limpeza:", payments["payment_type"].unique())

Removidas 3 linhas (not_defined + valor 0)
payment_installments == 0 após correção: 0
payment_type após limpeza: <StringArray>
['credit_card', 'boleto', 'voucher', 'debit_card']
Length: 4, dtype: str


## reviews

**Decisões:**
1. Converter `review_creation_date` e `review_answer_timestamp` para `datetime`.
2. **Não** remover/imputar os nulos em `review_comment_title`/`review_comment_message`: são campos opcionais -
   ausência de texto é uma resposta válida ("não comentou"), não um dado faltante por erro.
3. **Não** tratar os 814 `review_id` duplicados como duplicatas a remover: o par `(review_id, order_id)` é
   único, ou seja, cada linha é uma avaliação legítima de um pedido específico - é só o identificador de review
   que é reaproveitado entre pedidos diferentes, uma particularidade conhecida deste dataset. Documentamos que
   `review_id` não deve ser tratado como chave primária em etapas futuras.

In [5]:
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])
reviews.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

## products

**Decisões:**
1. `product_weight_g == 0` (4 linhas) → converter para `NaN`: peso zero é fisicamente impossível para um produto
   físico; melhor marcar como ausente do que manter um valor sabidamente errado.
2. Nulos em `product_category_name` e metadados associados (~610 linhas) e nas dimensões físicas de 2 produtos →
   **mantidos como `NaN`**: são produtos cadastrados de forma incompleta pelo vendedor; inventar categoria ou
   dimensão seria fabricar dado.
3. Padronizar `product_category_name` (`strip`) e acrescentar `product_category_name_english` via junção com
   `product_category_name_translation` - isso resolve a inconsistência de a categoria estar só em português.
   As 2 categorias sem tradução (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) ficam com
   `product_category_name_english` nulo; é uma lacuna da tabela de tradução original, documentada, não
   inventada.

In [6]:
products["product_category_name"] = products["product_category_name"].str.strip()

products.loc[products["product_weight_g"] == 0, "product_weight_g"] = np.nan

cat_translation["product_category_name"] = cat_translation["product_category_name"].str.strip()
products = products.merge(cat_translation, on="product_category_name", how="left")

print("product_weight_g == 0 após correção:", (products["product_weight_g"] == 0).sum())
print("categorias sem tradução após junção:",
      products.loc[products["product_category_name"].notna(), "product_category_name_english"].isna().sum())
products.head(3)

product_weight_g == 0 após correção: 0
categorias sem tradução após junção: 13


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure


## customers e sellers

**Decisão:** converter `customer_zip_code_prefix`/`seller_zip_code_prefix` de número inteiro para texto com 5
dígitos (zero-padded). CEP é um identificador categórico - tratá-lo como número faz o prefixo perder o zero à
esquerda em qualquer exibição/exportação (ex.: região de São Paulo, que começa com `0`), mesmo que o valor
numérico interno esteja correto. Estados já são 100% válidos (confirmado na avaliação) - nenhuma correção.

In [7]:
customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].astype(str).str.zfill(5)
sellers["seller_zip_code_prefix"] = sellers["seller_zip_code_prefix"].astype(str).str.zfill(5)

customers[["customer_zip_code_prefix"]].head(3)

,customer_zip_code_prefix
0,14409
1,09790
2,01151


## geolocation

**Decisões:**
1. Remover as ~261.831 linhas 100% duplicadas: não adicionam nenhuma informação nova e infladavam o arquivo em
   ~26%.
2. Remover as 31 linhas com latitude/longitude fora da caixa delimitadora aproximada do Brasil: são erros de
   geocodificação (outliers) que não podem representar um endereço real de um pedido brasileiro.
3. Converter `geolocation_city` claramente inválido (igual a uma sigla de estado, ou com entidades HTML /
   URL-encoding / pontuação solta - ~20 linhas no total) para `NaN`, em vez de manter texto corrompido.
4. **Não** normalizar as milhares de variações de grafia legítimas (com/sem acento, espaçamento) de nomes de
   cidade: exigiria uma tabela de referência externa de municípios, o que está fora do escopo do RA1. Isso é
   documentado como limitação conhecida - análises futuras que precisem de granularidade de cidade devem
   preferir `customer_city`/`seller_city` (já validados) ou o prefixo de CEP.
5. Converter `geolocation_zip_code_prefix` para texto com 5 dígitos, pelo mesmo motivo de `customers`/`sellers`.

In [8]:
linhas_antes = len(geo)
geo = geo.drop_duplicates().copy()
print(f"Removidas {linhas_antes - len(geo)} linhas duplicadas")

fora_do_brasil = (
    (geo["geolocation_lat"] > 6) | (geo["geolocation_lat"] < -34)
    | (geo["geolocation_lng"] > -32) | (geo["geolocation_lng"] < -74)
)
print(f"Removendo {fora_do_brasil.sum()} coordenadas fora do Brasil")
geo = geo[~fora_do_brasil].copy()

Removidas 261831 linhas duplicadas
Removendo 27 coordenadas fora do Brasil


In [9]:
estados_validos = {
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT","PA","PB","PE","PI",
    "PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO",
}
padrao_valido = re.compile(r"^[a-z0-9áéíóúãõâêôçàü\s\-\.\(\)/']*$")

cidade_e_sigla = geo["geolocation_city"].str.lower().isin({s.lower() for s in estados_validos})
cidade_com_caractere_invalido = ~geo["geolocation_city"].fillna("").str.match(padrao_valido)
cidade_invalida = cidade_e_sigla | cidade_com_caractere_invalido

print(f"Nulificando {cidade_invalida.sum()} valores de geolocation_city inválidos")
geo.loc[cidade_invalida, "geolocation_city"] = np.nan

geo["geolocation_zip_code_prefix"] = geo["geolocation_zip_code_prefix"].astype(str).str.zfill(5)

geo.shape

Nulificando 16 valores de geolocation_city inválidos


(738305, 5)

## Verificação final e exportação

In [10]:
tabelas_finais = {
    "olist_orders_dataset.csv": orders,
    "olist_customers_dataset.csv": customers,
    "olist_order_items_dataset.csv": items,
    "olist_order_payments_dataset.csv": payments,
    "olist_order_reviews_dataset.csv": reviews,
    "olist_products_dataset.csv": products,
    "olist_sellers_dataset.csv": sellers,
    "olist_geolocation_dataset.csv": geo,
    "product_category_name_translation.csv": cat_translation,
}

for nome, df in tabelas_finais.items():
    assert df.duplicated().sum() == 0, f"{nome} ainda tem duplicatas de linha inteira"

print("Nenhuma duplicata de linha inteira restante em nenhuma tabela.")
assert (payments["payment_installments"] == 0).sum() == 0
assert (products["product_weight_g"] == 0).sum() == 0
assert geo["geolocation_lat"].between(-34, 6).all()
assert geo["geolocation_lng"].between(-74, -32).all()
print("Checagens de valor inválido/outlier passaram.")

Nenhuma duplicata de linha inteira restante em nenhuma tabela.
Checagens de valor inválido/outlier passaram.


In [11]:
for nome, df in tabelas_finais.items():
    df.to_csv(PROCESSED / nome, index=False)

resumo = pd.DataFrame(
    [(nome, df.shape[0], df.shape[1]) for nome, df in tabelas_finais.items()],
    columns=["arquivo", "linhas", "colunas"],
)
resumo

,arquivo,linhas,colunas
0,olist_orders_dataset.csv,99441,8
1,olist_customers_dataset.csv,99441,5
2,olist_order_items_dataset.csv,112650,7
3,olist_order_payments_dataset.csv,103883,5
4,olist_order_reviews_dataset.csv,99224,7
5,olist_products_dataset.csv,32951,10
6,olist_sellers_dataset.csv,3095,4
7,olist_geolocation_dataset.csv,738305,5
8,product_category_name_translation.csv,71,2


## Dataset resultante do RA1

Os 9 arquivos tratados foram salvos em `data/processed/`, com os mesmos nomes e a mesma granularidade dos
originais (a única junção é a tradução das categorias de produto, que é uma padronização; agregações e outras junções são transformação e ficam para as próximas etapas). As principais mudanças
em relação a `data/raw/`:

- colunas de data/hora como `datetime` em vez de texto;
- 3 pagamentos inválidos (`not_defined` + valor 0) removidos; 2 corrigidos (`installments` 0 → 1);
- peso de produto 0 → `NaN` (4 linhas);
- categoria de produto traduzida para inglês via junção com a tabela de tradução;
- prefixos de CEP padronizados como texto de 5 dígitos em `customers`, `sellers` e `geolocation`;
- `geolocation` sem duplicatas (-26%), sem coordenadas fora do Brasil e sem nomes de cidade corrompidos.

Ver `docs/04_dicionario_dados.md` para o dicionário de dados completo de cada arquivo resultante, e
`docs/03_relatorio_qualidade.md` para o relatório de qualidade detalhado.